In [1]:
# 路径引导：无论从项目根还是 notebooks/ 目录启动内核，都能找到 src 包
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('项目根：', ROOT)

项目根： D:\agent项目集\端到端电商智能决策平台


# 01 数据质量探索

对 Olist 原始数据与数仓事实表做质量体检：行数、主键粒度、缺失率、金额口径。
完整结论见 `reports/data_quality_report.md` 与 `reports/data_dictionary.md`。

In [2]:
import pandas as pd
from src import config, sql_engine
pd.set_option('display.width', 120)

## 1. 原始文件行数与大小

In [3]:
rows = []
for table, filename in config.RAW_FILES.items():
    path = config.RAW_DIR / filename
    if not path.exists():
        continue
    n = sum(1 for _ in open(path, encoding='utf-8')) - 1
    rows.append({'table': table, 'file': filename, 'rows': n,
                 'size_mb': round(path.stat().st_size / 1e6, 1)})
pd.DataFrame(rows)

,table,file,rows,size_mb
0,raw_orders,olist_orders_dataset.csv,99441,17.7
1,raw_order_items,olist_order_items_dataset.csv,112650,15.0
2,raw_order_payments,olist_order_payments_dataset.csv,103886,5.6
3,raw_order_reviews,olist_order_reviews_dataset.csv,104719,14.5
4,raw_customers,olist_customers_dataset.csv,99441,8.6
5,raw_products,olist_products_dataset.csv,32951,2.4
6,raw_sellers,olist_sellers_dataset.csv,3095,0.2
7,raw_geolocation,olist_geolocation_dataset.csv,1000163,61.3
8,raw_category_translation,product_category_name_translation.csv,71,0.0


## 2. 数仓事实表粒度检查（一行一订单 / 一行一明细）

In [4]:
checks = {
    'fct_order': sql_engine.query_scalar('SELECT COUNT(*) FROM fct_order'),
    'fct_order 主键去重': sql_engine.query_scalar('SELECT COUNT(DISTINCT order_id) FROM fct_order'),
    'fct_order_item': sql_engine.query_scalar('SELECT COUNT(*) FROM fct_order_item'),
    'fct_order_item 主键去重': sql_engine.query_scalar(
        'SELECT COUNT(*) FROM (SELECT DISTINCT order_id, order_item_id FROM fct_order_item)'),
}
pd.Series(checks, name='行数')

fct_order               99441
fct_order 主键去重          99441
fct_order_item         112650
fct_order_item 主键去重    112650
Name: 行数, dtype: int64

> 两两相等说明建表过程中没有发生 fan-out（重复计数）。
> 这些数字已被 `tests/test_sql_contracts.py` 固化为回归断言。

## 3. 关键字段缺失率

In [5]:
missing = sql_engine.read_sql('''
SELECT
    AVG(CASE WHEN review_score IS NULL THEN 1.0 ELSE 0 END) AS review_missing,
    AVG(CASE WHEN delivered_ts IS NULL THEN 1.0 ELSE 0 END) AS delivered_missing,
    AVG(CASE WHEN payment_total IS NULL THEN 1.0 ELSE 0 END) AS payment_missing
FROM fct_order
''')
missing.T.rename(columns={0: '缺失率'}).round(4)

,缺失率
review_missing,0.0132
delivered_missing,0.0298
payment_missing,0.0000


## 4. 金额口径：支付总额 vs 商品 GMV vs 运费

In [6]:
amounts = sql_engine.read_sql('''
SELECT ROUND(SUM(payment_total), 2) AS payment,
       ROUND(SUM(product_gmv), 2) AS gmv,
       ROUND(SUM(freight_total), 2) AS freight
FROM fct_order
''')
amounts

,payment,gmv,freight
0,16008872.12,13591643.7,2256599.44


## 5. 不完整月份说明

Olist 2016-09/10/12 与 2018-09 之后的月份采集不完整，趋势类分析统一限定在
**2017-01-01 ~ 2018-08-31**（`src/config.py` 的 `ANALYSIS_START_DATE/END_DATE`）。
月度订单曲线上可以看到两端断崖，这不是业务真实变化。

In [7]:
monthly = sql_engine.read_sql('''
SELECT strftime('%Y-%m', purchase_ts) AS month, COUNT(*) AS orders
FROM fct_order GROUP BY month ORDER BY month
''')
monthly

,month,orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026
